# Pull new metrics from mccray
First based off 'Original Transcript'

In [61]:
import pandas as pd
import re, nltk, csv
from nltk.corpus import words, PlaintextCorpusReader

In [62]:
# pull entire dataset
# df = pd.read_csv("../data/mccray/october_sprint/mccray d_main.csv")

df = pd.read_csv("../data/mccray/october_sprint/D_mcc_main.csv")
print(f"df len: {len(df)}")
print(list(df))

# # nltk base
# # # nltk.download('words')
# english_vocab = set(w.lower() for w in words.words())

# improved ntlk corpus
wordslist = PlaintextCorpusReader('../data/mccray', 'combined_words_corpus.txt')
english_vocab = set(w.lower() for w in wordslist.words())

def save_csv(df, title):
    df.to_csv(title,
          index=False,
          encoding='utf-8',
          quoting=csv.QUOTE_NONNUMERIC,   
          escapechar='\\',
          lineterminator='\n')

print(len(df))

df len: 14427
['Title', 'Creator', 'Contributors', 'Date', 'Approximate Date', 'Source', 'Subject', 'Local Subject', 'S.C. County', 'Description', 'Extent', 'Digital Collection', 'Website', 'Contributing Institution', 'Rights', 'Time Period', 'Geographic Location', 'Language', 'Digitization Specifications', 'Date Digital', 'Type', 'Format', 'Media Type', 'Identifier', 'Note', 'Digital Assistant', 'OCLC number', 'Date created', 'Date modified', 'Reference URL', 'CONTENTdm number', 'CONTENTdm file name', 'CONTENTdm file path', 'Year', 'Original Transcript', 'Original Len', 'Special Pattern', 'General Pattern', 'Repeat Chars', 'Short/No Transcript', 'Quality', 'Issue Types', 'Detected Artifacts', 'Semi-clean Transcript', '% English (base ntlk)', '% Semi-clean English (base ntlk)', '% English', '% Semi-clean English']
14427


In [63]:
og_tr = "Original Transcript"
sc_tr = "Semi-clean Transcript"

title = 'Title'
descrip = 'Description'

# Function to apply re pattern
def re_pattern(text, pattern):
    if not isinstance(text, str) or len(text) == 0:
        return 0
    total = len(text)
    matches = len(pattern.findall(text))
    return (matches / total) * 100

# patterns
alphanum_standard = re.compile(r'[a-zA-Z0-9\s.,!?;:\'"()\-_/]')
alpha_only = re.compile(r'[a-zA-Z]')

# nltk en
def percent_real_english_nltk(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0
    tokens = re.findall(r"[A-Za-z]+", text.lower())
    if not tokens:
        return 0
    real = [t for t in tokens if t in english_vocab]
    return 100 * len(real) / len(tokens)


In [64]:
# Title 
df['% alphanum_standard title'] = df[title].apply(lambda x: re_pattern(x, alphanum_standard))
df['% alpha_only title'] = df[title].apply(lambda x: re_pattern(x, alpha_only))
df['% en title'] = df[title].apply(percent_real_english_nltk)

# Description
df['% alphanum_standard descrip'] = df[descrip].apply(lambda x: re_pattern(x, alphanum_standard))
df['% alpha_only descrip'] = df[descrip].apply(lambda x: re_pattern(x, alpha_only))
df['% en descrip'] = df[descrip].apply(percent_real_english_nltk)

In [65]:
# Transcripts
# alphanumeric + standard symbols
# define what is alphanumeric / standard symbols (essentially opposite of special pattern from ocr_cleaning)
df['% alphanum_standard D_mcc_raw tr'] = df[og_tr].apply(lambda x: re_pattern(x, alphanum_standard))
df['% alphanum_standard D_mcc_cleaned tr'] = df[sc_tr].apply(lambda x: re_pattern(x, alphanum_standard))

# alphabet a-Z
df['% alpha_only D_mcc_raw tr'] = df[og_tr].apply(lambda x: re_pattern(x, alpha_only))
df['% alpha_only D_mcc_cleaned tr'] = df[sc_tr].apply(lambda x: re_pattern(x, alpha_only))

# english words
df['% en D_mcc_raw tr'] = df[og_tr].apply(percent_real_english_nltk)
df['% en D_mcc_cleaned tr'] = df[sc_tr].apply(percent_real_english_nltk)

In [66]:
df[0:10]

,Title,Creator,Contributors,Date,Approximate Date,Source,Subject,Local Subject,S.C. County,Description,...,% en title,% alphanum_standard descrip,% alpha_only descrip,% en descrip,% alphanum_standard D_mcc_raw tr,% alphanum_standard D_mcc_cleaned tr,% alpha_only D_mcc_raw tr,% alpha_only D_mcc_cleaned tr,% en D_mcc_raw tr,% en D_mcc_cleaned tr
0,Afro-American Newsboy Application signed by Mr...,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,An application to be a Newsboy for the Afro-Am...,...,100.0,100.0,77.108434,100.0,100.000000,100.000000,79.133858,81.048387,94.594595,94.594595
1,Lighthouse Informer receipt,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A blank Lighthouse and Informer receipt.,...,100.0,100.0,85.000000,100.0,98.802395,98.648649,62.275449,70.270270,100.000000,100.000000
2,"The Lighthouse Solicitor's Record, Home Office...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,"A solicitor's record, home office order, and s...",...,100.0,100.0,80.681818,100.0,99.060150,98.927039,67.669173,77.253219,98.611111,98.611111
3,The Lighthouse Remittance Envelope(Front),NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,The front of a Lighthouse remittance envelope.,...,100.0,100.0,84.782609,100.0,100.000000,100.000000,63.333333,65.517241,100.000000,100.000000
4,"Advertisment, Chicken Special at the Pig Trail...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,An ad about the Chicken Special for the Pig Tr...,...,100.0,100.0,79.629630,100.0,100.000000,100.000000,64.351852,70.202020,87.878788,87.878788
5,"The Lighthouse Solicitor's Record, Home Office...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,"A solicitor's record, home office order, and s...",...,100.0,100.0,80.681818,100.0,98.880597,98.715203,67.164179,77.087794,98.611111,98.611111
6,The Lighthouse Prospectus,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A prospectus of the Lighthouse with permanent ...,...,100.0,100.0,86.206897,100.0,99.350649,99.334221,75.324675,77.230360,94.845361,94.845361
7,The Lighthouse Temporary and Permanent Operati...,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,Temporary and Permanent Operating Costs includ...,...,100.0,100.0,83.884298,100.0,99.382716,99.303136,59.567901,67.247387,85.714286,85.714286
8,"Letter, Lighthouse Newspaper Sign-up Sheet sen...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A sign-up sheet for the Lighthouse Newspaper s...,...,100.0,100.0,78.151261,100.0,100.000000,100.000000,74.609375,77.959184,100.000000,100.000000
9,"Letter, Lighthouse Newspaper Sign-up Sheet sen...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A sign-up sheet for the Lighthouse Newspaper s...,...,100.0,100.0,78.846154,100.0,100.000000,100.000000,77.254902,79.116466,100.000000,100.000000


In [67]:
# Save
save_csv(df, "new_metrics main -- all.csv")

In [68]:
# Averages
def column_stats(df, colname):
    """
    Returns basic statistics (min, mean, median, max, percentiles) for a numeric column.
    """
    if colname not in df.columns:
        raise ValueError(f"Column '{colname}' not found in DataFrame.")
    
    # Drop NaN values to avoid errors
    series = pd.to_numeric(df[colname], errors='coerce').dropna()

    if series.empty:
        return {"min": None, "p10": None, "p25": None, "mean": None, "median": None, "max": None}
    
    stats = {
        "min": series.min(),
        "p10": series.quantile(0.10),
        "p25": series.quantile(0.25),
        "mean": series.mean(),
        "median": series.median(),
        "max": series.max()
    }
    
    print(f"\nStats for column: **{colname}**")
    print("-" * 40)
    print(f"Minimum:        {stats['min']:.2f}")
    print(f"10th percentile: {stats['p10']:.2f}")
    print(f"25th percentile: {stats['p25']:.2f}")
    print(f"Mean:           {stats['mean']:.2f}")
    print(f"Median:         {stats['median']:.2f}")
    print(f"Maximum:        {stats['max']:.2f}")
    print("-" * 40)

    return stats

check_cols = [
    "% alphanum_standard title",
    "% alpha_only title",
    "% en title",
    "% alphanum_standard descrip",
    "% alpha_only descrip",
    "% en descrip",
    "% alphanum_standard D_mcc_raw tr",
    "% alphanum_standard D_mcc_cleaned tr",
    "% alpha_only D_mcc_raw tr",
    "% alpha_only D_mcc_cleaned tr",
    "% en D_mcc_raw tr",
    "% en D_mcc_cleaned tr"
]

for col in check_cols:
    column_stats(df, col)

print()


Stats for column: **% alphanum_standard title**
----------------------------------------
Minimum:        78.57
10th percentile: 100.00
25th percentile: 100.00
Mean:           99.95
Median:         100.00
Maximum:        100.00
----------------------------------------

Stats for column: **% alpha_only title**
----------------------------------------
Minimum:        0.00
10th percentile: 61.90
25th percentile: 66.33
Mean:           72.34
Median:         72.58
Maximum:        100.00
----------------------------------------

Stats for column: **% en title**
----------------------------------------
Minimum:        0.00
10th percentile: 100.00
25th percentile: 100.00
Mean:           99.76
Median:         100.00
Maximum:        100.00
----------------------------------------

Stats for column: **% alphanum_standard descrip**
----------------------------------------
Minimum:        0.00
10th percentile: 100.00
25th percentile: 100.00
Mean:           98.63
Median:         100.00
Maximum:      